In [34]:
import requests
import pandas as pd
#bibli qui permet de communiquer avec un service via HTTP

In [6]:
url = "http://localhost:11434/api/generate"

data = {
    "model": "llama3.2:3b",
    "prompt": "Explain what machine learning is in simple terms.",
    "stream": False
}

In [7]:
response = requests.post(url, json=data)

In [8]:
result = response.json()
print(result)

{'model': 'llama3.2:3b', 'created_at': '2026-09-30T21:55:14.301104Z', 'response': 'Machine learning is a way for computers to learn and improve on their own without being explicitly programmed.\n\nImagine you\'re trying to teach a child to recognize different types of animals. You show them pictures of cats, dogs, and birds, and explain what makes each one unique. At first, the child might struggle to tell the difference between a cat and a dog. But with more practice and exposure, they start to recognize patterns and can make educated guesses.\n\nMachine learning works in a similar way. You give computers a large dataset of examples (like pictures of animals) and tell them what\'s important (like "this is a cat" or "this is a dog"). Then, the computer starts to look for patterns and relationships between the examples, and uses that information to make predictions or take actions on its own.\n\nThere are different types of machine learning, but some common examples include:\n\n* Image 

In [9]:
result["eval_count"]
tokens_generated = result["eval_count"]

In [10]:
tokens_generated = result["eval_count"]

In [11]:
result["eval_duration"]

7118174000

In [12]:
generation_time = result["eval_duration"]
print (generation_time)

7118174000


In [13]:
generation_time = result["eval_duration"]/ 1e9
print (generation_time)

7.118174


In [14]:
token_per_second = tokens_generated / generation_time
print(token_per_second) 

35.26185226716852


In [15]:
def benchmark(prompt):
    data = {
        "model": "llama3.2:3b",
        "prompt": prompt,
        "stream": False

    }

    response = requests.post(url, json=data)
    result = response.json()
    
    tokens_generated = result["eval_count"]
    generation_time = result["eval_duration"] /1e9
    token_per_second = tokens_generated / generation_time

    return tokens_generated, generation_time, token_per_second
    

In [16]:
benchmark("Explain what artificial intelligence is in simple terms.")

(249, 7.479146, 33.29257110370622)

In [17]:
speeds = []

for i in range(5):
    result = benchmark("Explain what artificial intelligence is in simple terms.")
    speeds.append(result[2])

print(speeds)

[34.265502989838545, 34.04838729813562, 35.69748831795465, 32.80870808957164, 32.58148442311277]


In [18]:
average_speed = sum(speeds)/len(speeds)
print(average_speed)

33.88031422372264


### Premier benchmark

Le même prompt a été exécuté 5 fois afin de limiter l'effet des variations entre les générations.

Prompt testé :
"Explain what artificial intelligence is in simple terms."

Vitesse moyenne observée : **33.88 tokens/s**

Cette valeur constitue une première mesure de référence pour Llama 3.2 3B en quantification Q4_K_M sur ma machine.

### Benchmark sur différents types de tâches

Le premier benchmark a été réalisé avec un seul prompt. Afin d'obtenir une mesure plus représentative des performances du modèle, nous allons maintenant tester plusieurs types de tâches.

Quatre catégories de prompts seront utilisées :
- explication d'un concept ;
- raisonnement ;
- génération de texte ;
- génération de code.

Chaque prompt sera exécuté plusieurs fois afin de calculer une vitesse moyenne en tokens par seconde et de limiter l'effet des variations entre les générations.

In [19]:
prompts = [
    "Explain what a neural network is in simple terms.",
    "A train travels 120 km in 1.5 hours and then 80 km in 1 hour. What is its average speed over the whole journey? Explain your reasoning.",
    "Write a 10-line summary about the life and work of Carl Friedrich Gauss.",
    "Write a Python loop for i from 0 to 5 that computes f(x) = x + 2."
]

In [20]:
for i in range(len(prompts)):
    result = benchmark(prompts[i])
    print(result)

(296, 8.777689, 33.72186004767314)
(185, 5.685997, 32.53607063106083)
(248, 8.446709999, 29.360543931230097)
(251, 9.024269, 27.813887196846636)


In [28]:
for i in range(len(prompts)):
    speeds = []

    for j in range(5):
        result = benchmark(prompts[i])
        speeds.append(result[2])

    average_speed = sum(speeds) / len(speeds)
    print(average_speed)

28.803929581789003
32.71744444754451
34.202604975341885
33.72680505718284


In [30]:
average_speeds = []

for i in range(len(prompts)):
    speeds = []

    for j in range(5):
        result = benchmark(prompts[i])
        speeds.append(result[2])

    average_speed = sum(speeds) / len(speeds)
    average_speeds.append(average_speed)

In [32]:
print(average_speeds)

[32.9252315813189, 35.06163345544603, 34.804467951688764, 34.48974340957583]


In [47]:
tableau = pd.DataFrame({"Prompts": prompts, "Average speed": average_speeds})
print(round(tableau, 2))
#print(round(tableau["Average speed"], 2))

                                             Prompts  Average speed
0  Explain what a neural network is in simple terms.          32.93
1  A train travels 120 km in 1.5 hours and then 8...          35.06
2  Write a 10-line summary about the life and wor...          34.80
3  Write a Python loop for i from 0 to 5 that com...          34.49


#### Les résultats montrent que la vitesse de génération reste relativement stable selon le type de tâche, avec une moyenne comprise entre 32,93 et 35,06 tokens/s. Sur les prompts testés, le type de tâche semble donc avoir peu d’influence sur la vitesse de génération de Llama 3.2 3B Q4_K_M.